In [ ]:
import pandas as pd
import numpy as np
import iqplot
from plot_tools import *
import itertools as it
import bokeh.plotting
import bokeh.io
import holoviews as hv
from holoviews import dim, opts
import bokeh.models
from plot_tools import *
from bokeh.layouts import gridplot
from scipy.spatial.distance import jensenshannon
hv.extension('bokeh')

In [ ]:
df_coal = pd.read_csv('e003_coalescence_metadata_round4_good.csv')
df_abun = pd.read_csv('e003_coalescence_metadata_round4_abundances.csv')
df_abun['AC']=df_abun['parent_subjects'].transform(lambda x: 'AC' in x)
df_abun=df_abun.loc[~df_abun['AC'],:]
df_abun_get_alpha = df_abun.loc[df_abun['relative_abundance']>=1e-3,:]#.drop(columns='Unnamed:0')
df_abun_get_alpha['counts']=1
df_abun_get_alpha = df_abun_get_alpha.groupby(['sample','mesocosm','passage','inoculumn_sample','type_mesocosm']).sum(numeric_only=True).reset_index()
df_abun_get_alpha['subjects'] = df_abun_get_alpha['type_mesocosm'].transform(lambda x: '-'.join(x.split('-')[:2]))
df_abun_get_alpha['env'] = df_abun_get_alpha['type_mesocosm'].transform(lambda x: '-'.join(x.split('-')[2:]))
bad_samples = df_abun_get_alpha.loc[df_abun_get_alpha['counts']<20,:]
bad_samples
df_abund = df_abun.loc[~df_abun['sample'].isin(bad_samples),:]
df_abun['Lineage'].values[0]
df_abun['order']= df_abun['Lineage'].transform(lambda x: x.split(';')[3])
df_abun
df_abun.loc[df_abun['family']=='f__','family']= df_abun.loc[df_abun['family']=='f__','order'] + ' ' + df_abun.loc[df_abun['family']=='f__','family']
#df_abun = df_abun.loc[df_abun['family']!='family',:]
#df_abun['family'].unique()
df_abun = df_abun.groupby(['sample','family','parent_media','parent_subjects','media','type_mesocosm','mesocosm','passage']).sum(numeric_only=True).reset_index()
df_abun.head()

In [ ]:
df = pd.read_csv('e003_coalescence_metadata_round4_good.csv')
df = df[['comm', 'comm-media', 'inoculumn',
       'inoculumn_sample', 'is_inoculumn', 'media', 'mesocosm',
       'parent_media', 'parent_subjects', 'passage', 'sample',
       'type_mesocosm', 'redo', 'counts', 'mesocosm-passage', 'replicate']]

df.loc[df['mesocosm']=='A3-AA-AC/PP-mBHI-mGAM','inoculumn_sample']='A3-e003Coalescence-mBHI-inoculumn-redo'
df.loc[df['mesocosm']=='D3-AC/PP-AF-mBHI-mGAM','inoculumn_sample']='D3-e003Coalescence-Inoculumn-mBHI'
df.loc[df['mesocosm']=='E3-AA-AC/PP-mBHI-mBHI','inoculumn_sample']='A3-e003Coalescence-mBHI-inoculumn-redo'
df.loc[df['mesocosm']=='B10-AC/PP-AE-mGAM-mGAM','inoculumn_sample']='B4-e003Coalescence-Inoculumn-mGAM'
df.loc[df['mesocosm']=='B10-AC/PP-AE-mGAM-mBHI','inoculumn_sample']='B4-e003Coalescence-Inoculumn-mGAM'
df.loc[df['mesocosm']=='B4-AC/PP-AE-mBHI-mGAM','inoculumn_sample']='B4-e003Coalescence-Inoculumn-mBHI'
df.loc[df['mesocosm']=='B8-AA-AC/PP-mGAM-mGAM','inoculumn_sample']='B2-e003Coalescence-Inoculumn-mGAM'
df.loc[df['mesocosm']=='B11-AC/PP-AF-mGAM-mBHI','inoculumn_sample']='B11-AC/PP-AF-mGAM-mBHI'
df.loc[df['mesocosm']=='F10-AC/PP-AE-mGAM-mBHI','inoculumn_sample']='B4-e003Coalescence-Inoculumn-mGAM'

for meso in df['mesocosm'].unique():
    in_sample = df.loc[df['mesocosm']==meso,'inoculumn_sample'].unique()
    if len(in_sample)>1:
        print(meso)

#df.to_csv('e003_coalescence_metadata_round4_good.csv',index=False)

In [ ]:
def Renkonen_sim(s1_abuns,s2_abuns):
    good_arrays = np.zeros((len(s2_abuns),2))
    good_arrays[:,0]=s1_abuns
    good_arrays[:,1]=s2_abuns
    mins = np.min(good_arrays,axis=1)
    return np.sum(mins)

In [ ]:
df_abun_p7 = df_abun.loc[df_abun['passage'] == 7,:]
sample1s = []
sample2s = []
JSDs = []
for sample1,sample2 in it.combinations(df_abun_p7['sample'].unique(),2):
    sample1s.append(sample1)
    sample2s.append(sample2)
    s1_abuns = df_abun.loc[df_abun['sample'] == sample1,:].sort_values(by='family')
    #print(len(s1_abuns))
    s2_abuns  = df_abun.loc[df_abun['sample'] == sample2,:].sort_values(by='family')
   # JSDs.append(jensenshannon(s1_abuns['relative_abundance'].values, s2_abuns['relative_abundance'].values))
    JSDs.append(Renkonen_sim(s1_abuns['relative_abundance'].values, s2_abuns['relative_abundance'].values))

In [ ]:
df_pairwise_JSD = pd.DataFrame(data = {'sample1': sample1s, 'sample2': sample2s, 
                                       'JSD': JSDs, })
df_pairwise_JSD['type_mesocosm1']=np.nan
df_pairwise_JSD['type_mesocosm2']=np.nan
for type_meso in df_abun['type_mesocosm'].unique():
    samples_meso=df_abun.loc[df_abun['type_mesocosm']==type_meso,'sample'].unique()
    df_pairwise_JSD.loc[df_pairwise_JSD['sample1'].isin(samples_meso),'type_mesocosm1']=type_meso
    df_pairwise_JSD.loc[df_pairwise_JSD['sample2'].isin(samples_meso),'type_mesocosm2']=type_meso

In [ ]:
df_pairwise_JSD['subjects1'] = df_pairwise_JSD['type_mesocosm1'].transform(lambda x: '-'.join(x.split('-')[:2]))
df_pairwise_JSD['subjects2'] =df_pairwise_JSD['type_mesocosm2'].transform(lambda x: '-'.join(x.split('-')[:2]))

df_pairwise_JSD['env1'] = df_pairwise_JSD['type_mesocosm1'].transform(lambda x: '-'.join(x.split('-')[2:]))
df_pairwise_JSD['env2'] = df_pairwise_JSD['type_mesocosm2'].transform(lambda x: '-'.join(x.split('-')[2:]))

In [ ]:
df_pairwise_JSD['parent_media1']=df_pairwise_JSD['env1'].transform(lambda x: x.split('-')[0])
df_pairwise_JSD['parent_media2']=df_pairwise_JSD['env2'].transform(lambda x: x.split('-')[0])
df_pairwise_JSD['coal_media1']=df_pairwise_JSD['env1'].transform(lambda x: x.split('-')[1])
df_pairwise_JSD['coal_media2']=df_pairwise_JSD['env2'].transform(lambda x: x.split('-')[1])

In [ ]:
df_pairwise_JSD['same_subjects'] = df_pairwise_JSD['subjects1'] == df_pairwise_JSD['subjects2'] 
df_pairwise_JSD['same_coal_media'] = df_pairwise_JSD['coal_media1'] == df_pairwise_JSD['coal_media2'] 
df_pairwise_JSD['same_parent_media'] = df_pairwise_JSD['parent_media1'] == df_pairwise_JSD['parent_media2'] 
df_pairwise_JSD['same_env'] = df_pairwise_JSD['env1'] == df_pairwise_JSD['env2'] 
df_pairwise_JSD_only_double = df_pairwise_JSD#.loc[~df_pairwise_JSD['subjects1'].isin(['AA-AA','AC/PP-AC/PP',
                                                                                   #   'AE-AE','AF-AF']),:]

#df_pairwise_JSD_only_double = df_pairwise_JSD_only_double.loc[~df_pairwise_JSD_only_double['subjects2'].isin(['AA-AA','AC/PP-AC/PP',
 #                                                                                     'AE-AE','AF-AF']),:]

In [ ]:
#df_pairwise_JSD.to_csv('pairwise_JSD.csv')

In [ ]:
#df_pairwise_JSD_only_double['JSD']

In [ ]:
df_pairwise_JSD_only_double['comparison']='Diff Subject-Diff Parent Media - Diff Coal Media'
df_pairwise_JSD_only_double.loc[df_pairwise_JSD_only_double['same_subjects'],'comparison']= 'Same Subject-Diff Parent Media - Diff Coal Media'
df_pairwise_JSD_only_double.loc[(df_pairwise_JSD_only_double['same_subjects'])*(df_pairwise_JSD_only_double['same_parent_media']),
    'comparison']= 'Same Subject-Same Parent Media - Diff Coal Media'
df_pairwise_JSD_only_double.loc[(df_pairwise_JSD_only_double['same_subjects'])*(df_pairwise_JSD_only_double['same_coal_media']),
    'comparison']= 'Same Subject-Diff Parent Media - Same Coal Media'
df_pairwise_JSD_only_double.loc[(df_pairwise_JSD_only_double['same_subjects'])*(df_pairwise_JSD_only_double['same_env']),
    'comparison']= 'Same Subject-Same Parent Media - Same Coal Media'

df_pairwise_JSD_only_double.loc[(~df_pairwise_JSD_only_double['same_subjects'])*(df_pairwise_JSD_only_double['same_parent_media']),
    'comparison']= 'Diff Subject-Same Parent Media - Diff Coal Media'
df_pairwise_JSD_only_double.loc[(~df_pairwise_JSD_only_double['same_subjects'])*(df_pairwise_JSD_only_double['same_coal_media']),
    'comparison']= 'Diff Subject-Diff Parent Media - Same Coal Media'
df_pairwise_JSD_only_double.loc[(~df_pairwise_JSD_only_double['same_subjects'])*(df_pairwise_JSD_only_double['same_env']),
    'comparison']= 'Diff Subject-Same Parent Media - Same Coal Media'

df_pairwise_JSD_only_double_double = df_pairwise_JSD_only_double.loc[~df_pairwise_JSD_only_double['subjects1'].isin(['AA-AA','AC/PP-AC/PP',
                                                                                      'AE-AE','AF-AF']),:]

df_pairwise_JSD_only_double_double = df_pairwise_JSD_only_double_double.loc[~df_pairwise_JSD_only_double_double['subjects2'].isin(['AA-AA','AC/PP-AC/PP',
                                                                                     'AE-AE','AF-AF']),:]

p = iqplot.strip(df_pairwise_JSD_only_double_double, 
                 q = 'JSD', cats = ['comparison'], spread='jitter',palette = bokeh.palettes.Colorblind[8],
                 order = ['Diff Subject-Diff Parent Media - Diff Coal Media',
                          'Diff Subject-Same Parent Media - Diff Coal Media',
                          'Diff Subject-Diff Parent Media - Same Coal Media',
                          'Diff Subject-Same Parent Media - Same Coal Media',
                          'Same Subject-Diff Parent Media - Diff Coal Media',
                          'Same Subject-Same Parent Media - Diff Coal Media',
                          'Same Subject-Diff Parent Media - Same Coal Media',
                          'Same Subject-Same Parent Media - Same Coal Media'],
               #  palette = [bokeh.palettes.Colorblind[7][0],bokeh.palettes.Colorblind[7][4],
                #            bokeh.palettes.Colorblind[7][-2],bokeh.palettes.Colorblind[7][1],],
                width =650,height=500,
                 show_legend=False,q_axis='y')

p.ygrid.grid_line_color = None
p.y_range = bokeh.models.Range1d(0,1)
p.yaxis.axis_label = 'Renkonen similarity'
p.xaxis.major_label_orientation = 4*np.pi/10
p.yaxis.axis_label_text_font_style='normal'
#p.yaxis.axis_label_text_style='normal'
p.output_backend = 'svg'
export_plot_pdf(p, 'Renkonen similarity_media_subj')
p.xaxis.axis_label = 'Same Media - Same Subject'
p.output_backend='svg'
#export_plot_pdf(p,')
bokeh.io.show(p)

In [ ]:
from scipy.stats import permutation_test

def statistic(x, y, axis):
    return np.median(x, axis=axis) - np.median(y, axis=axis)

from scipy.stats import permutation_test
p1s = []
p2s = []
pvalues = []
stats = []
compares = df_pairwise_JSD_only_double['comparison'].unique()
compares = ['Diff Subject-Same Parent Media - Diff Coal Media','Same Subject-Same Parent Media - Same Coal Media',
           'Diff Subject-Same Parent Media - Same Coal Media']
for p1,p2 in it.combinations(compares,2):
    x = df_pairwise_JSD_only_double_double.loc[df_pairwise_JSD_only_double_double['comparison']==p1,'JSD'].values
    y = df_pairwise_JSD_only_double_double.loc[df_pairwise_JSD_only_double_double['comparison']==p2,'JSD'].values
    res = permutation_test((x, y), statistic, vectorized=True,permutation_type='independent',
                       n_resamples=10000, alternative='two-sided')
   # print(p,res)
    p1s.append(p1)
    p2s.append(p2)
    passage.append(p)
    pvalues.append(res.pvalue)
df = pd.DataFrame(data = {'compare1':p1s,'compare2':p2s,'pval':pvalues, })
df['sig']=df['pval']<.05
df['sigsig']=df['pval']<.001
df#.loc[~df['sigsig'],:]

In [ ]:
np.isnan(x)

In [ ]:
df_pairwise_JSD_only_double_double['replicates']='0'
df_pairwise_JSD_only_double_double.loc[df_pairwise_JSD_only_double_double['type_mesocosm1']==df_pairwise_JSD_only_double_double['type_mesocosm2'],'replicates']='2'

p = iqplot.strip(df_pairwise_JSD_only_double_double.sort_values(by='replicates'), q = 'JSD',cats = 'replicates', spread='jitter',
                 #color_column = 'sub-media',
                 palette = [bokeh.palettes.Colorblind[7][1]]*4,
                height= 300,width=400,
                 x_axis_label=None,
                 show_legend=True,q_axis='y')

p.xaxis.major_label_orientation = np.pi/3
p.ygrid.grid_line_color = None
p.y_range = bokeh.models.Range1d(0,1)
p.legend.visible=False
p.yaxis.axis_label_text_font_style='normal'
p.xaxis.axis_label_text_font_style='normal'
p.yaxis.axis_label_text_font_size='25px'
p.yaxis.major_label_text_font_size='20px'
#p.xaxis.major_label_text_font_size='20px'
#p.xaxis.axis_label_text_font_size='25px'
#p.xaxis.axis_label = 'Same Media - Same Subject'
bokeh.io.show(p)
p.output_backend='svg'
export_plot_pdf(p,'jsd_2')




In [ ]:
### from scipy.spatial.distance import jensenshannon
mesos = []
subs1jsd = []
subs2jsd = []
maxJSD = []
ino_JSD = []
ins_all = df_abun.loc[df_abun['passage']==0,:]
df_abunp7 = df_abun.loc[df_abun['passage']==7,:]
for meso in df_abunp7['mesocosm'].unique():
    _,parent_subject1,parent_subject2,parent_media,_ = meso.split('-')
    
    spog_df = df_abunp7.loc[df_abunp7['mesocosm'] == meso,:].sort_values(by='species_id')
    
    ins = ins_all.loc[ins_all['parent_media']==parent_media,:]
    ins1 = ins.loc[ins['parent_subjects'] == parent_subject1 + '-' +  parent_subject1,'sample'].values[0]
    ins2 = ins.loc[ins['parent_subjects'] == parent_subject2 + '-' +  parent_subject2,'sample'].values[0]
 #   ins3 = ins.loc[ins['parent_subjects'] == parent_subject1 + '-' +  parent_subject2,'sample'].values[0]
    ins3 = ins.loc[ins['parent_subjects'] == parent_subject1 + '-' +  parent_subject2,'sample'].values[0]
    if ins1 == 'A2-e003Coalescence-Inoculumn-mBHI':
        ins1 = 'A2-e003Coalescence-mBHI-inoculumn-redo'

  #  print(ins1,ins2)
    sp1 = df_abun.loc[df_abun['sample'] == ins1,:].sort_values(by='species_id')
    sp2 = df_abun.loc[df_abun['sample'] == ins2,:].sort_values(by='species_id')
    sp3 = df_abun.loc[df_abun['sample'] == ins3,:].sort_values(by='species_id')
  #  sp3 = df_abundance.loc[df_abundance['sample'] == ins3,:].sort_values(by='species_id')

    JSD1 = jensenshannon(spog_df['relative_abundance'].values,sp1['relative_abundance'].values)
    JSD2 = jensenshannon(spog_df['relative_abundance'].values,sp2['relative_abundance'].values)
    JSD3 = jensenshannon(spog_df['relative_abundance'].values,sp3['relative_abundance'].values)
    subs1jsd.append(JSD1)
    subs2jsd.append(JSD2)
    maxJSD.append(np.max([JSD1,JSD2]))
    ino_JSD.append(JSD3)
    
                  
    mesos.append(meso)

df_p0 = pd.DataFrame(data = {'JSD':subs1jsd+subs2jsd})
df_po = pd.DataFrame(data = {'JSD':ino_JSD})
df_p0['replicates']='1'

dfplot = pd.concat([df_p0,df_pairwise_JSD_only_double_double[['JSD','replicates']]])

p = iqplot.strip(dfplot.sort_values(by='replicates'), q = 'JSD',cats = 'replicates', spread='jitter',
                 #color_column = 'sub-media',
                 palette = [bokeh.palettes.Colorblind[7][1]]*4,
                height= 150,width=200,
                 x_axis_label=None,marker_kwargs =dict(alpha=.5),
                 show_legend=True,q_axis='y')

qs = dfplot.groupby(["replicates"]).JSD.quantile([0.25, 0.5, 0.75])
qs = qs.unstack().reset_index()
qs.columns = ["replicates","q1", "q2", "q3"]
df = pd.merge(dfplot, qs, on=["replicates"], how="left")
source = bokeh.models.ColumnDataSource(df)
p.vbar("replicates", 0.4, "q2", "q3", source=source, color=None, line_color="grey",line_alpha=.5)
p.vbar("replicates", 0.4, "q1", "q2", source=source, color=None, line_color="grey",line_alpha=.5)
p.legend.visible=False

p.xaxis.axis_label_text_font_size='22px'
p.xaxis.major_label_text_font_size='15px'
p.yaxis.axis_label_text_font_size='22px'
p.yaxis.major_label_text_font_size='15px'
p.xaxis.minor_tick_line_color= None
p.xgrid.grid_line_color = None
p.ygrid.grid_line_color = None
p.yaxis.minor_tick_line_color= None
p.yaxis.axis_label='JSD'
#p.y_range = bokeh.models.Range1d(20,70)
p.xaxis.axis_label='Timepoint'
p.xaxis.axis_label_text_font_style='normal'
p.yaxis.axis_label_text_font_style='normal'
p.y_range = bokeh.models.Range1d(0,1)
p.legend.visible = False

p.output_backend='svg'
#p.legend.visible=False
export_plot_pdf(p,'JSD')
bokeh.io.show(p)
#bokeh.io.show(p)

In [ ]:
qs

In [ ]:
from scipy.stats import permutation_test

def statistic(x, y, axis):
    return np.median(x, axis=axis) - np.median(y, axis=axis)

from scipy.stats import permutation_test
passage = []
pvalues = []
stats = []
for p in dfplot['replicates'].unique():
    x = dfplot.loc[dfplot['replicates']=='0','JSD'].values
    y = dfplot.loc[dfplot['replicates']==p,'JSD'].values
    res = permutation_test((x, y), statistic, vectorized=True,permutation_type='independent',
                       n_resamples=10000, alternative='two-sided')
   # print(p,res)
    passage.append(p)
    pvalues.append(res.pvalue)
df = pd.DataFrame(data = {'passage':passage,'pval':pvalues})
df.sort_values(by='passage')

In [ ]:
x